In [1]:
import pandas as pd

In [2]:
# import data
schedules = pd.read_csv('Data/schedules.csv')
player_stats = pd.read_csv('Data/player_stats.csv')
team_stats = pd.read_csv('Data/team_stats.csv')

In [3]:
schedules.head()

,game_id,season,game_type,week,gameday,weekday,gametime,away_team,away_score,home_team,...,wind,away_qb_id,home_qb_id,away_qb_name,home_qb_name,away_coach,home_coach,referee,stadium_id,stadium
0,1999_01_MIN_ATL,1999,REG,1,1999-09-12,Sunday,NaN,MIN,17,ATL,...,NaN,00-0003761,00-0002876,Randall Cunningham,Chris Chandler,Dennis Green,Dan Reeves,Gerry Austin,ATL00,Georgia Dome
1,1999_01_KC_CHI,1999,REG,1,1999-09-12,Sunday,NaN,KC,17,CHI,...,12.0,00-0006300,00-0010560,Elvis Grbac,Shane Matthews,Gunther Cunningham,Dick Jauron,Phil Luckett,CHI98,Soldier Field
2,1999_01_PIT_CLE,1999,REG,1,1999-09-12,Sunday,NaN,PIT,43,CLE,...,12.0,00-0015700,00-0004230,Kordell Stewart,Ty Detmer,Bill Cowher,Chris Palmer,Bob McElwee,CLE00,Cleveland Browns Stadium
3,1999_01_OAK_GB,1999,REG,1,1999-09-12,Sunday,NaN,OAK,24,GB,...,10.0,00-0005741,00-0005106,Rich Gannon,Brett Favre,Jon Gruden,Ray Rhodes,Tony Corrente,GNB00,Lambeau Field
4,1999_01_BUF_IND,1999,REG,1,1999-09-12,Sunday,NaN,BUF,14,IND,...,NaN,00-0005363,00-0010346,Doug Flutie,Peyton Manning,Wade Phillips,Jim Mora,Ron Blum,IND99,RCA Dome


In [4]:
# create dataframe of team records by year, have to account for them being both home and away
import numpy as np

# 1. Determine the game outcomes from the perspective of home and away
schedules["Home_Win"] = np.where(schedules["home_score"] > schedules["away_score"], 1, 0)
schedules["Home_Loss"] = np.where(schedules["home_score"] < schedules["away_score"], 1, 0)
schedules["Away_Win"] = schedules["Home_Loss"]  # If home lost, away won
schedules["Away_Loss"] = schedules["Home_Win"]  # If home won, away lost

# Handle ties (if applicable to your sport)
schedules["Tie"] = np.where(schedules["home_score"] == schedules["away_score"], 1, 0)

# 2. Aggregate Home Records
home_rec = (
    schedules.groupby(["season", "home_team"])[["Home_Win", "Home_Loss", "Tie"]]
    .sum()
    .reset_index()
)
home_rec.rename(
    columns={
        "home_team": "Team",
        "Home_Win": "Wins",
        "Home_Loss": "Losses",
        "Tie": "Ties",
    },
    inplace=True,
)

# 3. Aggregate Away Records
away_rec = (
    schedules.groupby(["season", "away_team"])[["Away_Win", "Away_Loss", "Tie"]]
    .sum()
    .reset_index()
)
away_rec.rename(
    columns={
        "away_team": "Team",
        "Away_Win": "Wins",
        "Away_Loss": "Losses",
        "Tie": "Ties",
    },
    inplace=True,
)

# 4. Combine and Sum
# Concatenate them vertically, then group by Season and Team to sum them up
season_records = (
    pd.concat([home_rec, away_rec])
    .groupby(["season", "Team"])[["Wins", "Losses", "Ties"]]
    .sum()
    .reset_index()
)

# Optional: Add a Win Percentage column for cleaner analysis
total_games = season_records["Wins"] + season_records["Losses"] + season_records["Ties"]
season_records["Win_Pct"] = (
    (season_records["Wins"] + 0.5 * season_records["Ties"]) / total_games
).round(3)

season_records.head()

,season,Team,Wins,Losses,Ties,Win_Pct
0,1999,ARI,6,10,0,0.375
1,1999,ATL,5,11,0,0.312
2,1999,BAL,8,8,0,0.500
3,1999,BUF,11,6,0,0.647
4,1999,CAR,8,8,0,0.500


In [7]:
# save records to CSV
season_records.to_csv("Data/team_records.csv", index=False)

In [6]:
# pull just regular season schedules and do the same as above

schedules_reg = schedules[schedules["game_type"] == 'REG']

#1. Determine the game outcomes from the perspective of home and away
schedules_reg["Home_Win"] = np.where(schedules_reg["home_score"] > schedules_reg["away_score"], 1, 0)
schedules_reg["Home_Loss"] = np.where(schedules_reg["home_score"] < schedules_reg["away_score"], 1, 0)
schedules_reg["Away_Win"] = schedules_reg["Home_Loss"]  # If home lost, away won
schedules_reg["Away_Loss"] = schedules_reg["Home_Win"]  # If home won, away lost

# Handle ties (if applicable to your sport)
schedules_reg["Tie"] = np.where(schedules_reg["home_score"] == schedules_reg["away_score"], 1, 0)

# 2. Aggregate Home Records
home_rec = (
    schedules_reg.groupby(["season", "home_team"])[["Home_Win", "Home_Loss", "Tie"]]
    .sum()
    .reset_index()
)
home_rec.rename(
    columns={
        "home_team": "Team",
        "Home_Win": "Wins",
        "Home_Loss": "Losses",
        "Tie": "Ties",
    },
    inplace=True,
)

# 3. Aggregate Away Records
away_rec = (
    schedules_reg.groupby(["season", "away_team"])[["Away_Win", "Away_Loss", "Tie"]]
    .sum()
    .reset_index()
)
away_rec.rename(
    columns={
        "away_team": "Team",
        "Away_Win": "Wins",
        "Away_Loss": "Losses",
        "Tie": "Ties",
    },
    inplace=True,
)

# 4. Combine and Sum
# Concatenate them vertically, then group by Season and Team to sum them up
season_records_reg = (
    pd.concat([home_rec, away_rec])
    .groupby(["season", "Team"])[["Wins", "Losses", "Ties"]]
    .sum()
    .reset_index()
)

# Optional: Add a Win Percentage column for cleaner analysis
total_games = season_records_reg["Wins"] + season_records_reg["Losses"] + season_records_reg["Ties"]
season_records_reg["Win_Pct"] = (
    (season_records_reg["Wins"] + 0.5 * season_records_reg["Ties"]) / total_games
).round(3)

season_records_reg.head()

,season,Team,Wins,Losses,Ties,Win_Pct
0,1999,ARI,6,10,0,0.375
1,1999,ATL,5,11,0,0.312
2,1999,BAL,8,8,0,0.500
3,1999,BUF,11,5,0,0.688
4,1999,CAR,8,8,0,0.500


In [7]:
season_records_reg.to_csv("Data/team_regular_season_records.csv", index=False)

## Creating a Master Dataset

In [11]:
player_stats['position'].value_counts()

position
WR     5647
LB     4140
DE     3914
RB     3906
DB     3497
CB     3210
DT     3187
TE     3162
OT     2648
G      2575
QB     2080
OLB    1898
C      1351
S      1281
K      1099
FS     1091
P       984
FB      720
LS      709
ILB     705
SAF     659
MLB     581
NT      509
DL       38
OL        7
Name: count, dtype: int64

In [19]:
# join QB, RB, WR, TE stats to team regular season records

master_stats = player_stats[player_stats['position'].isin(['QB', 'RB', 'WR', 'TE', 'FB'])]

combined_data = pd.merge(season_records_reg, master_stats, how='left', left_on=["season", "Team"], right_on=["season", "recent_team"])

pd.set_option('display.max_columns', None)
combined_data.head()

,season,Team,Wins,Losses,Ties,Win_Pct,player_id,player_name,player_display_name,position,position_group,headshot_url,season_type,recent_team,games,completions,attempts,passing_yards,passing_tds,passing_interceptions,sacks_suffered,sack_yards_lost,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_cpoe,passing_2pt_conversions,pacr,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_first_downs,rushing_epa,rushing_2pt_conversions,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles,receiving_fumbles_lost,receiving_air_yards,receiving_yards_after_catch,receiving_first_downs,receiving_epa,receiving_2pt_conversions,racr,target_share,air_yards_share,wopr,special_teams_tds,def_tackles_solo,def_tackles_with_assist,def_tackle_assists,def_tackles_for_loss,def_tackles_for_loss_yards,def_fumbles_forced,def_sacks,def_sack_yards,def_qb_hits,def_interceptions,def_interception_yards,def_pass_defended,def_tds,def_fumbles,def_safeties,misc_yards,fumble_recovery_own,fumble_recovery_yards_own,fumble_recovery_opp,fumble_recovery_yards_opp,fumble_recovery_tds,penalties,penalty_yards,punt_returns,punt_return_yards,kickoff_returns,kickoff_return_yards,fg_made,fg_att,fg_missed,fg_blocked,fg_long,fg_pct,fg_made_0_19,fg_made_20_29,fg_made_30_39,fg_made_40_49,fg_made_50_59,fg_made_60_,fg_missed_0_19,fg_missed_20_29,fg_missed_30_39,fg_missed_40_49,fg_missed_50_59,fg_missed_60_,fg_made_list,fg_missed_list,fg_blocked_list,fg_made_distance,fg_missed_distance,fg_blocked_distance,pat_made,pat_att,pat_missed,pat_blocked,pat_pct,gwfg_made,gwfg_att,gwfg_missed,gwfg_blocked,gwfg_distance_list,fantasy_points,fantasy_points_ppr
0,1999,ARI,6,10,0,0.375,00-0000871,M.Bates,Mario Bates,RB,RB,https://static.www.nfl.com/image/private/f_aut...,REG,ARI,16.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,NaN,72.0,202.0,9.0,0.0,0.0,23.0,2.726209,0.0,5.0,7.0,34.0,0.0,0.0,0.0,0.0,0.0,2.0,-0.809647,0.0,0.000000,0.012987,0.000000,0.019481,0.0,5.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,10.0,0.0,0.0,52.0,1231.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,NaN,77.60,82.60
1,1999,ARI,6,10,0,0.375,00-0001532,D.Boston,David Boston,WR,WR,https://static.www.nfl.com/image/private/f_aut...,REG,ARI,15.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,NaN,5.0,0.0,0.0,0.0,0.0,0.0,-2.571011,0.0,40.0,81.0,473.0,2.0,1.0,0.0,33.0,0.0,21.0,1.172459,0.0,14.333333,0.150278,0.063462,0.269841,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,3.0,20.0,7.0,62.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,NaN,59.30,99.30
2,1999,ARI,6,10,0,0.375,00-0001907,D.Brown,Dave Brown,QB,QB,https://static.www.nfl.com/image/private/f_aut...,REG,ARI,8.0,84.0,169.0,944.0,2.0,6.0,18.0,-130.0,4.0,1.0,96.0,944.0,39.0,-40.548629,NaN,0.0,9.833333,13.0,49.0,0.0,0.0,0.0,5.0,4.327390,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,NaN,36.66,36.66
3,1999,ARI,6,10,0,0.375,00-0001919,D.Brown,Derek Brown,TE,TE,https://static.www.nfl.com/image/private/f_aut...,REG,ARI,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.836658,0.0,NaN,0.001855,0.000000,0.002783,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0

In [20]:
# get rid of unnecessary columns and save to CSV
combined_data.drop(columns=[
    'recent_team',
    'player_id',
    'headshot_url',
    'season_type',
    'sacks_suffered',
    'sack_yards_lost',
    'sack_fumbles',
    'sack_fumbles_lost',
    'passing_air_yards',
    'passing_yards_after_catch',
    'passing_first_downs',
    'passing_cpoe',
    'passing_2pt_conversions',
    'pacr',
    'rushing_first_downs',
    'rushing_2pt_conversions',
    'receiving_first_downs',
    'receiving_2pt_conversions',
    'racr',
    'air_yards_share'
], inplace=True)

# drop columns starting with def_, punt_, fg_, kickoff_, pat_, gwfg_, fantasy_, fumble_
combined_data.drop(columns=[
    col for col in combined_data.columns if col.startswith(('def_', 'punt_', 'fg_', 'kickoff_', 'pat_', 'gwfg_', 'fantasy_', 'fumble_'))
    ], inplace=True)
combined_data.head()

,season,Team,Wins,Losses,Ties,Win_Pct,player_name,player_display_name,position,position_group,games,completions,attempts,passing_yards,passing_tds,passing_interceptions,passing_epa,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_epa,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles,receiving_fumbles_lost,receiving_air_yards,receiving_yards_after_catch,receiving_epa,target_share,wopr,special_teams_tds,misc_yards,penalties,penalty_yards
0,1999,ARI,6,10,0,0.375,M.Bates,Mario Bates,RB,RB,16.0,0.0,0.0,0.0,0.0,0.0,NaN,72.0,202.0,9.0,0.0,0.0,2.726209,5.0,7.0,34.0,0.0,0.0,0.0,0.0,0.0,-0.809647,0.012987,0.019481,0.0,0.0,1.0,10.0
1,1999,ARI,6,10,0,0.375,D.Boston,David Boston,WR,WR,15.0,0.0,0.0,0.0,0.0,0.0,NaN,5.0,0.0,0.0,0.0,0.0,-2.571011,40.0,81.0,473.0,2.0,1.0,0.0,33.0,0.0,1.172459,0.150278,0.269841,0.0,0.0,3.0,20.0
2,1999,ARI,6,10,0,0.375,D.Brown,Dave Brown,QB,QB,8.0,84.0,169.0,944.0,2.0,6.0,-40.548629,13.0,49.0,0.0,0.0,0.0,4.327390,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.000000,0.000000,0.0,0.0,0.0,0.0
3,1999,ARI,6,10,0,0.375,D.Brown,Derek Brown,TE,TE,2.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,NaN,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.836658,0.001855,0.002783,0.0,0.0,1.0,4.0
4,1999,ARI,6,10,0,0.375,M.Cody,Mac Cody,WR,WR,13.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,NaN,6.0,11.0,60.0,1.0,0.0,0.0,6.0,0.0,0.659355,0.020408,0.038689,0.0,0.0,1.0,0.0


In [21]:
combined_data.to_csv("Data/combined_team_player_stats.csv", index=False)

## What Correlates the Most to Win % Out of These Stats?

In [22]:
# get correlation matrix to win % for each position group

rbs = combined_data[combined_data['position_group'] == 'RB']
wrs = combined_data[combined_data['position_group'] == 'WR']
qbs = combined_data[combined_data['position_group'] == 'QB']
tes = combined_data[combined_data['position_group'] == 'TE']

In [24]:
corr_rb = rbs.corr(numeric_only=True)['Win_Pct'].sort_values(ascending=False)
print(corr_rb)

Win_Pct                        1.000000
Wins                           0.997381
receiving_epa                  0.167835
rushing_tds                    0.148740
passing_epa                    0.137211
rushing_epa                    0.128839
games                          0.123296
receiving_tds                  0.110442
rushing_yards                  0.096698
carries                        0.081204
receiving_yards                0.059231
penalties                      0.052413
penalty_yards                  0.049695
receiving_yards_after_catch    0.044615
receptions                     0.039714
receiving_air_yards            0.035691
targets                        0.025981
special_teams_tds              0.021072
completions                    0.020142
passing_yards                  0.019975
receiving_fumbles              0.019369
receiving_fumbles_lost         0.016938
passing_tds                    0.016005
rushing_fumbles                0.009755
season                         0.004200


In [25]:
corr_wr = wrs.corr(numeric_only=True)['Win_Pct'].sort_values(ascending=False)
print(corr_wr)

Win_Pct                        1.000000
Wins                           0.997137
receiving_epa                  0.210451
receiving_tds                  0.183026
games                          0.175176
receiving_yards                0.134352
receptions                     0.120438
receiving_yards_after_catch    0.116153
rushing_epa                    0.082773
penalty_yards                  0.072116
penalties                      0.071230
targets                        0.070604
receiving_air_yards            0.062829
receiving_fumbles              0.058934
passing_epa                    0.058883
rushing_yards                  0.048979
wopr                           0.044851
rushing_tds                    0.044699
target_share                   0.044015
receiving_fumbles_lost         0.037817
carries                        0.034624
special_teams_tds              0.034471
rushing_fumbles                0.013328
passing_interceptions          0.010717
rushing_fumbles_lost           0.006767


In [27]:
qbs['TD/INT'] = qbs['passing_tds'] / qbs['passing_interceptions']
corr_qb = qbs.corr(numeric_only=True)['Win_Pct'].sort_values(ascending=False)
print(corr_qb)

Win_Pct                        1.000000
Wins                           0.997442
passing_epa                    0.463605
TD/INT                         0.316786
passing_tds                    0.241431
passing_yards                  0.166478
games                          0.165241
carries                        0.156660
completions                    0.143543
attempts                       0.120271
rushing_tds                    0.113526
rushing_yards                  0.066973
penalties                      0.053606
penalty_yards                  0.044689
receiving_fumbles              0.019440
rushing_fumbles                0.009983
receiving_yards_after_catch    0.008760
receiving_fumbles_lost         0.008308
rushing_fumbles_lost           0.003995
receiving_tds                 -0.000601
receptions                    -0.001856
receiving_epa                 -0.002258
season                        -0.006784
targets                       -0.010185
passing_interceptions         -0.016450


In [28]:
corr_te = tes.corr(numeric_only=True)['Win_Pct'].sort_values(ascending=False)
print(corr_te)

Win_Pct                        1.000000
Wins                           0.997144
receiving_epa                  0.195089
receiving_tds                  0.172342
games                          0.127699
receiving_yards                0.104402
receiving_yards_after_catch    0.092097
receptions                     0.085085
rushing_epa                    0.071512
passing_epa                    0.069304
receiving_air_yards            0.065786
targets                        0.060814
penalties                      0.054901
penalty_yards                  0.043260
wopr                           0.034939
target_share                   0.033612
passing_tds                    0.032555
rushing_tds                    0.024659
carries                        0.023920
passing_yards                  0.023015
rushing_yards                  0.021764
completions                    0.021622
rushing_fumbles                0.021524
attempts                       0.021140
receiving_fumbles_lost         0.016539


There aren't many strong correlations, so we will do what we can

The columns for use will be:
* tds
* yards
* epa
* TD/INT OR carries OR target_share

In [30]:
# calculate total PF and PA for each team in a given season, then calculate Pythagorean win percentage for a baseline statistic
# 1. Calculate stats for when the team is playing at HOME
home_stats = (
    schedules_reg.groupby(["season", "home_team"])[["home_score", "away_score"]]
    .sum()
    .reset_index()
)
home_stats.rename(
    columns={
        "home_team": "team",
        "home_score": "PF",  # Points scored at home
        "away_score": "PA",  # Points allowed at home
    },
    inplace=True,
)

# 2. Calculate stats for when the team is playing AWAY
away_stats = (
    schedules_reg.groupby(["season", "away_team"])[["away_score", "home_score"]]
    .sum()
    .reset_index()
)
away_stats.rename(
    columns={
        "away_team": "team",
        "away_score": "PF",  # Points scored on the road
        "home_score": "PA",  # Points allowed on the road
    },
    inplace=True,
)

# 3. Concatenate home and away stats, then sum them together
team_season_points = (
    pd.concat([home_stats, away_stats])
    .groupby(["season", "team"])[["PF", "PA"]]
    .sum()
    .reset_index()
)

# 4. Optional: Calculate your Pythagorean Base Team Strength right here!
exponent = 2.37
team_season_points["Base_Team_Strength"] = (
    team_season_points["PF"] ** exponent
) / (team_season_points["PF"] ** exponent + team_season_points["PA"] ** exponent)

team_season_points.head()

,season,team,PF,PA,Base_Team_Strength
0,1999,ARI,245,382,0.258714
1,1999,ATL,285,380,0.335858
2,1999,BAL,324,277,0.591807
3,1999,BUF,320,229,0.688475
4,1999,CAR,421,381,0.558877


In [34]:
# merge team_season_points to combined_data to get Base_Team_Strength as well
team_season_points.rename(
    columns={"team": "Team"}, inplace=True
)

combined_data = pd.merge(
    combined_data,
    team_season_points[["season", "Team", "Base_Team_Strength"]],
    on=["season", "Team"],
    how="left",
)

combined_data.head()

,season,Team,Wins,Losses,Ties,Win_Pct,player_name,player_display_name,position,position_group,games,completions,attempts,passing_yards,passing_tds,passing_interceptions,passing_epa,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_epa,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles,receiving_fumbles_lost,receiving_air_yards,receiving_yards_after_catch,receiving_epa,target_share,wopr,special_teams_tds,misc_yards,penalties,penalty_yards,Base_Team_Strength
0,1999,ARI,6,10,0,0.375,M.Bates,Mario Bates,RB,RB,16.0,0.0,0.0,0.0,0.0,0.0,NaN,72.0,202.0,9.0,0.0,0.0,2.726209,5.0,7.0,34.0,0.0,0.0,0.0,0.0,0.0,-0.809647,0.012987,0.019481,0.0,0.0,1.0,10.0,0.258714
1,1999,ARI,6,10,0,0.375,D.Boston,David Boston,WR,WR,15.0,0.0,0.0,0.0,0.0,0.0,NaN,5.0,0.0,0.0,0.0,0.0,-2.571011,40.0,81.0,473.0,2.0,1.0,0.0,33.0,0.0,1.172459,0.150278,0.269841,0.0,0.0,3.0,20.0,0.258714
2,1999,ARI,6,10,0,0.375,D.Brown,Dave Brown,QB,QB,8.0,84.0,169.0,944.0,2.0,6.0,-40.548629,13.0,49.0,0.0,0.0,0.0,4.327390,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.000000,0.000000,0.0,0.0,0.0,0.0,0.258714
3,1999,ARI,6,10,0,0.375,D.Brown,Derek Brown,TE,TE,2.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,NaN,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.836658,0.001855,0.002783,0.0,0.0,1.0,4.0,0.258714
4,1999,ARI,6,10,0,0.375,M.Cody,Mac Cody,WR,WR,13.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,NaN,6.0,11.0,60.0,1.0,0.0,0.0,6.0,0.0,0.659355,0.020408,0.038689,0.0,0.0,1.0,0.0,0.258714


In [35]:
combined_data.to_csv("Data/combined_team_player_stats.csv", index=False)
team_season_points.to_csv("Data/team_season_points.csv", index=False)

In [ ]:
# validate number of teams per season in combined_data to make sure the merges didn't lose teams
unique_teams_by_season = (
    combined_data.groupby("season")["Team"].nunique().reset_index()
)

# Rename columns for clarity
unique_teams_by_season.columns = ["Season", "Unique_Teams_Count"]

# Display the result
print(unique_teams_by_season)

    Season  Unique_Teams_Count
0     1999                  31
1     2000                  31
2     2001                  31
3     2002                  32
4     2003                  32
5     2004                  32
6     2005                  32
7     2006                  32
8     2007                  32
9     2008                  32
10    2009                  32
11    2010                  32
12    2011                  32
13    2012                  32
14    2013                  32
15    2014                  32
16    2015                  32
17    2016                  32
18    2017                  32
19    2018                  32
20    2019                  32
21    2020                  32
22    2021                  32
23    2022                  32
24    2023                  32
25    2024                  32
26    2025                  32
